# Stacking

We now train stage-2 models using the outputs from our stage-1 models (linear, XGB, LGB, CatBoost from `03_Baselines`). We explore two techniques: (1) use stage-1 OOF as the **target** — stage 2 trains on the residuals (errors) of the stage-1 ensemble; and (2) use stage-1 OOF as a **feature** — append the OOFs as columns to the stage-2 training data. We finish with a simple logistic-regression meta-blender on the OOFs alone and write the submission.

Everything is on CPU and scored by AUC. Our test set has no labels, so we report CV OOF AUC and produce a submission file rather than a test score.

In [ ]:
VER = 1

## Load Data

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

train = pd.read_parquet('data/train_features.parquet')
y_train_true = train['PitNextLap'].astype(int).values
print('Train shape:', train.shape)
train.head(1)

In [ ]:
test = pd.read_parquet('data/test_features.parquet')
print('Test shape:', test.shape)
test.head(1)

Load the stage-1 OOF (train) and PRED (test) arrays saved by `03_Baselines`. The test preds were stored as `(N_SPLITS, len(test))` and are averaged over folds here.

In [ ]:
oof_linear = np.load(f'data/train_oof_linear_v{VER}.npy')
oof_xgb    = np.load(f'data/train_oof_xgb_v{VER}.npy')
oof_lgb    = np.load(f'data/train_oof_lgb_v{VER}.npy')
oof_cb     = np.load(f'data/train_oof_cb_v{VER}.npy')

pred_linear = np.load(f'data/test_pred_linear_v{VER}.npy').mean(0)
pred_xgb    = np.load(f'data/test_pred_xgb_v{VER}.npy').mean(0)
pred_lgb    = np.load(f'data/test_pred_lgb_v{VER}.npy').mean(0)
pred_cb     = np.load(f'data/test_pred_cb_v{VER}.npy').mean(0)

for name, oof in [('linear', oof_linear), ('xgb', oof_xgb), ('lgb', oof_lgb), ('cb', oof_cb)]:
    print(f'stage-1 {name:7s} OOF AUC = {roc_auc_score(y_train_true, oof):.5f}')

## Cross-Validation Setup

Stage-2 reuses the same 10-fold `StratifiedGroupKFold` grouped by `(Race, Year)` so val folds align with the stage-1 OOF folds. We stack on the **static** engineered features only (no per-fold target encodings) — that signal is already baked into the stage-1 OOFs.

In [ ]:
N_SPLITS = 10
RANDOM_STATE = 42
TARGET = 'PitNextLap'

groups = train.groupby(['Race', 'Year']).ngroup().values
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Static features (everything 03_Baselines used EXCEPT the per-fold target encodings).
RAW_NUMERIC = ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position',
               'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
               'RaceProgress', 'Position_Change']
ENGINEERED  = ['stint_progress', 'laps_remaining_frac', 'p_one_stop',
               'pit_window_cdf', 'compound_stint_pit_rate', 'prev_lap_time_delta',
               'laptime_tenths', 'laptime_hundredths', 'laptime_thousandths',
               'tyrelife_is_integer', 'nearest_real_pitnextlap', 'nearest_real_distance',
               'lap_time_pct_in_lap', 'same_compound_drivers_this_lap']
STATIC = RAW_NUMERIC + ENGINEERED + ['Race']
print(f'static features for stacking: {len(STATIC)}')

## Stacking - Use OOF as Target (residual)

Our stage-1 ensemble is the mean of the four OOFs. We train an XGBoost **regressor** to predict the residual `y - base`, then add the base prediction back. The model learns the patterns the ensemble still gets wrong. (Final values may fall slightly outside [0, 1], which is fine — AUC only depends on ranking.)

In [ ]:
from xgboost import XGBRegressor

# Stage-1 ensemble = mean of the four base OOFs / test preds
base_oof  = np.column_stack([oof_linear, oof_xgb, oof_lgb, oof_cb]).mean(1)
base_pred = np.column_stack([pred_linear, pred_xgb, pred_lgb, pred_cb]).mean(1)
print(f'base (mean of 4) OOF AUC = {roc_auc_score(y_train_true, base_oof):.5f}')

xgb_params = dict(
    n_estimators=2000, max_depth=5, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, objective='reg:squarederror',
    tree_method='hist', enable_categorical=True,
    random_state=RANDOM_STATE, early_stopping_rounds=50,
)

X      = train[STATIC].copy()
X_test = test[STATIC].copy()
X['Race']      = X['Race'].astype('category')
X_test['Race'] = pd.Categorical(X_test['Race'], categories=X['Race'].cat.categories)
y = y_train_true - base_oof   # continuous residual target

oof_stack  = np.zeros(len(train))
pred_stack = np.zeros((N_SPLITS, len(test)))

for fold, (tr, va) in enumerate(cv.split(X, y_train_true, groups)):
    model = XGBRegressor(**xgb_params)
    model.fit(X.iloc[tr], y[tr], eval_set=[(X.iloc[va], y[va])], verbose=500)
    oof_stack[va]    = model.predict(X.iloc[va])
    pred_stack[fold] = model.predict(X_test)
    fold_auc = roc_auc_score(y_train_true[va], base_oof[va] + oof_stack[va])
    print(f'Fold {fold+1} AUC: {fold_auc:.5f}  best_iter={model.best_iteration}')

# Add the base prediction back
oof_stack  += base_oof
pred_stack += base_pred

print('-' * 40)
print(f'Residual-stack, CV OOF AUC: {roc_auc_score(y_train_true, oof_stack):.5f}')

In [ ]:
# XGB importance (gain). Keys are column names since we pass a DataFrame.
score = model.get_booster().get_score(importance_type='gain')
imp = pd.DataFrame({'feature': list(score.keys()), 'importance': list(score.values())})\
        .sort_values('importance', ascending=False)
plt.figure(figsize=(6, 7))
plt.barh(imp['feature'], imp['importance'])
plt.gca().invert_yaxis()
plt.title('Residual-Stack XGB Feature Importance (gain)')
plt.tight_layout()
plt.show()

In [ ]:
np.save(f'data/train_oof_stack_target_v{VER}.npy', oof_stack)
np.save(f'data/test_pred_stack_target_v{VER}.npy', pred_stack)

## Stacking - Use OOF as Feature

Now we append the four stage-1 OOFs as feature columns and train an XGBoost **classifier** on `static features + OOFs`. When we plot importance, the OOF columns should dominate — the model leans on the stage-1 predictions and uses the raw features to adjust at the margins.

In [ ]:
from xgboost import XGBClassifier

OOF_COLS = ['oof_linear', 'oof_xgb', 'oof_lgb', 'oof_cb']
Xf      = train[STATIC].copy()
Xf_test = test[STATIC].copy()
Xf['Race']      = Xf['Race'].astype('category')
Xf_test['Race'] = pd.Categorical(Xf_test['Race'], categories=Xf['Race'].cat.categories)
for col, oof, pred in zip(OOF_COLS, [oof_linear, oof_xgb, oof_lgb, oof_cb],
                          [pred_linear, pred_xgb, pred_lgb, pred_cb]):
    Xf[col]      = oof
    Xf_test[col] = pred

clf_params = dict(
    n_estimators=2000, max_depth=5, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, objective='binary:logistic',
    tree_method='hist', enable_categorical=True, eval_metric='auc',
    random_state=RANDOM_STATE, early_stopping_rounds=50, verbosity=0,
)

oof_stack2  = np.zeros(len(train))
pred_stack2 = np.zeros((N_SPLITS, len(test)))

for fold, (tr, va) in enumerate(cv.split(Xf, y_train_true, groups)):
    model = XGBClassifier(**clf_params)
    model.fit(Xf.iloc[tr], y_train_true[tr], eval_set=[(Xf.iloc[va], y_train_true[va])], verbose=500)
    oof_stack2[va]    = model.predict_proba(Xf.iloc[va])[:, 1]
    pred_stack2[fold] = model.predict_proba(Xf_test)[:, 1]
    print(f'Fold {fold+1} AUC: {roc_auc_score(y_train_true[va], oof_stack2[va]):.5f}  best_iter={model.best_iteration}')

print('-' * 40)
print(f'Feature-stack, CV OOF AUC: {roc_auc_score(y_train_true, oof_stack2):.5f}')

In [ ]:
score = model.get_booster().get_score(importance_type='gain')
imp = pd.DataFrame({'feature': list(score.keys()), 'importance': list(score.values())})\
        .sort_values('importance', ascending=False)
plt.figure(figsize=(6, 7))
plt.barh(imp['feature'], imp['importance'])
plt.gca().invert_yaxis()
plt.title('Feature-Stack XGB Feature Importance (gain)')
plt.tight_layout()
plt.show()

In [ ]:
np.save(f'data/train_oof_stack_feature_v{VER}.npy', oof_stack2)
np.save(f'data/test_pred_stack_feature_v{VER}.npy', pred_stack2)

## Meta-Blender and Model Comparison

Finally, a logistic-regression meta-learner trained on the four stage-1 OOFs alone (the canonical level-2 blender). Honest OOF via `cross_val_predict`. We compare every approach by CV OOF AUC — the final submission is produced by a later notebook in the pipeline.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

X_meta      = np.column_stack([oof_linear, oof_xgb, oof_lgb, oof_cb])
X_meta_test = np.column_stack([pred_linear, pred_xgb, pred_lgb, pred_cb])

meta = LogisticRegression(C=1.0, max_iter=2000)
meta_oof = cross_val_predict(meta, X_meta, y_train_true, cv=5, method='predict_proba', n_jobs=-1)[:, 1]
meta.fit(X_meta, y_train_true)
meta_pred = meta.predict_proba(X_meta_test)[:, 1]
print('meta coefficients:', dict(zip(['linear', 'xgb', 'lgb', 'cb'], np.round(meta.coef_[0], 3))))

np.save(f'data/train_oof_stack_meta_v{VER}.npy', meta_oof)
np.save(f'data/test_pred_stack_meta_v{VER}.npy', meta_pred[None, :])

In [ ]:
# Compare every candidate by CV OOF AUC.
candidates = {
    'mean_blend':     base_oof,
    'residual_stack': oof_stack,
    'feature_stack':  oof_stack2,
    'meta_blend':     meta_oof,
}
scores = {k: roc_auc_score(y_train_true, v) for k, v in candidates.items()}
summary = pd.DataFrame({'OOF AUC': scores}).sort_values('OOF AUC', ascending=False)
print(summary.to_string())